# Standalone Kaggle Notebook for Person ReID

Notebook nay la file duy nhat de upload len Kaggle va train model theo dung cau hinh dang cho ket qua tot trong repo hien tai.

No tu chua:
- config
- dataset loader
- model
- loss
- train loop
- evaluate loop
- checkpoint saving
- metrics saving
- train.log va evaluate.log

Ban chi can sua cell config o ben duoi cho dung dataset path trong Kaggle roi chay lan luot.

Luu y quan trong:
- `configs/dadnet.yaml` hien tai trong repo dang dung `model.variant: vit`.
- Vi vay notebook nay da duoc chinh lai de dung backbone ViT-B/16 thay vi ban DADNet-ResNet cu.

In [ ]:
from __future__ import annotations

import json
import math
import random
import warnings
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch import amp
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.models import ResNet50_Weights, ViT_B_16_Weights, resnet50, vit_b_16
from tqdm.auto import tqdm

## 1. Config

Sua `DATASET_NAME` va `DATASET_ROOT` theo dataset ban da attach vao notebook Kaggle.

Vi du:
- Market1501: `/kaggle/input/market1501-dataset/Market-1501-v15.09.15`
- DukeMTMC-reID: `/kaggle/input/dukemtmcreid`
- MSMT17: `/kaggle/input/msmt17-dataset/MSMT17_V1`

In [ ]:
DATASET_NAME = "market1501"
DATASET_ROOT = "/kaggle/input/market1501-dataset/Market-1501-v15.09.15"
RUN_NAME = None

# Dat = 1 neu muon smoke nhanh tren Kaggle.
# Dat = None neu muon dung so epoch trong config.
EPOCHS_OVERRIDE = None

CONFIG = {
    "project_name": "person-reid-mlops",
    "experiment_name": "market1501-dadnet",
    "seed": 42,
    "device": "auto",
    "num_workers": 2,
    "data": {
        "dataset": {
            "name": DATASET_NAME,
        },
        "location": {
            "root": DATASET_ROOT,
            "splits": {
                "train": "bounding_box_train",
                "query": "query",
                "gallery": "bounding_box_test",
            },
            "list_files": {
                "train": "list_train.txt",
                "query": "list_query.txt",
                "gallery": "list_gallery.txt",
            },
        },
        "image_height": 224,
        "image_width": 224,
        "batch_size": 8,
        "eval_batch_size": 32,
        "instances_per_identity": 4,
    },
    "model": {
        "variant": "vit",
        "backbone": "vit_b_16",
        "pretrained": True,
        "embedding_dim": 512,
        "dropout": 0.1,
        "use_local_branch": True,
        "num_local_stripes": 4,
        "local_branch_dropout": 0.1,
    },
    "train": {
        "epochs": 50,
        "learning_rate": 0.00005,
        "backbone_lr_factor": 0.5,
        "weight_decay": 0.0005,
        "ce_weight": 1.0,
        "triplet_weight": 1.2,
        "triplet_margin": 0.35,
        "center_loss_weight": 0.0,
        "center_loss_lr": 0.25,
        "label_smoothing": 0.02,
        "scheduler_type": "cosine",
        "warmup_epochs": 5,
        "min_lr_scale": 0.02,
        "grad_clip_norm": 1.0,
        "amp": True,
        "early_stopping": {
            "enabled": True,
            "patience": 12,
            "min_delta": 0.001,
            "monitor": "mAP",
        },
    },
    "augmentation": {
        "color_jitter": True,
        "random_erasing": True,
        "random_grayscale_p": 0.0,
        "random_affine_degrees": 5.0,
        "random_occlusion_p": 0.15,
    },
    "evaluation": {
        "use_rerank": True,
        "rerank_k1": 20,
        "rerank_k2": 6,
        "rerank_lambda": 0.3,
        "flip_test": False,
    },
}

if EPOCHS_OVERRIDE is not None:
    CONFIG["train"]["epochs"] = int(EPOCHS_OVERRIDE)

dataset_slug = DATASET_NAME.lower().replace("_", "-")
if RUN_NAME is None:
    RUN_NAME = f"{dataset_slug}-dadnet-kaggle-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

ARTIFACT_ROOT = Path("/kaggle/working/artifacts") / dataset_slug / RUN_NAME
CHECKPOINTS_DIR = ARTIFACT_ROOT / "checkpoints"
METRICS_DIR = ARTIFACT_ROOT / "metrics"
LOGS_DIR = ARTIFACT_ROOT / "logs"

for path in (CHECKPOINTS_DIR, METRICS_DIR, LOGS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2))
print("Artifact root:", ARTIFACT_ROOT)

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def infer_device(device_name: str) -> torch.device:
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_name)


def save_json(payload: dict, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


class FileLogger:
    def __init__(self, path: Path) -> None:
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text("", encoding="utf-8")

    def log(self, message: str) -> None:
        print(message)
        with self.path.open("a", encoding="utf-8") as handle:
            handle.write(message + "\n")

In [ ]:
class ImageFolderReIDDataset(Dataset):
    def __init__(self, folder: str | Path, transform=None, relabel: bool = False) -> None:
        self.folder = Path(folder)
        self.transform = transform
        self.relabel = relabel
        self.samples = []

        pid_container = set()
        for image_path in sorted(self.folder.glob("*.jpg")):
            pid = int(image_path.name.split("_")[0])
            if pid == -1:
                continue
            pid_container.add(pid)

        self.pid2label = {pid: idx for idx, pid in enumerate(sorted(pid_container))}

        for image_path in sorted(self.folder.glob("*.jpg")):
            pid = int(image_path.name.split("_")[0])
            if pid == -1:
                continue
            camid = int(image_path.name.split("_")[1][1]) - 1
            mapped_pid = self.pid2label[pid] if relabel else pid
            self.samples.append({
                "img_path": str(image_path),
                "pid": mapped_pid,
                "camid": camid,
            })

        self.num_classes = len(pid_container)
        self.labels = [int(sample["pid"]) for sample in self.samples]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict[str, object]:
        sample = self.samples[index]
        image = Image.open(sample["img_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return {
            "image": image,
            "pid": int(sample["pid"]),
            "camid": int(sample["camid"]),
            "path": str(sample["img_path"]),
        }


class MSMT17Dataset(Dataset):
    SPLIT_DIRS = {
        "train": "train",
        "query": "test",
        "gallery": "test",
    }

    def __init__(self, root: str | Path, split: str, transform=None, relabel: bool = False) -> None:
        self.root = Path(root)
        self.split = split
        self.transform = transform
        self.relabel = relabel
        self.samples = []

        list_path = self.root / CONFIG["data"]["location"]["list_files"][split]
        split_dir = self.root / self.SPLIT_DIRS[split]

        pid_container = set()
        raw_samples = []

        for line in list_path.read_text(encoding="utf-8").splitlines():
            stripped = line.strip()
            if not stripped:
                continue
            relative_path_str, pid_str = stripped.split()
            pid = int(pid_str)
            if pid == -1:
                continue
            relative_path = Path(relative_path_str)
            image_path = split_dir / relative_path
            name_parts = relative_path.stem.split("_")
            camid = int(name_parts[2]) - 1
            pid_container.add(pid)
            raw_samples.append((image_path, pid, camid))

        self.pid2label = {pid: idx for idx, pid in enumerate(sorted(pid_container))}
        for image_path, pid, camid in raw_samples:
            mapped_pid = self.pid2label[pid] if relabel else pid
            self.samples.append({
                "img_path": str(image_path),
                "pid": mapped_pid,
                "camid": camid,
            })

        self.num_classes = len(pid_container)
        self.labels = [int(sample["pid"]) for sample in self.samples]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict[str, object]:
        sample = self.samples[index]
        image = Image.open(sample["img_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return {
            "image": image,
            "pid": int(sample["pid"]),
            "camid": int(sample["camid"]),
            "path": str(sample["img_path"]),
        }


def get_dataset_class(dataset_name: str):
    key = dataset_name.strip().lower()
    if key == "msmt17":
        return MSMT17Dataset
    return ImageFolderReIDDataset


def build_dataset(split: str, transform=None, relabel: bool = False):
    dataset_name = CONFIG["data"]["dataset"]["name"].strip().lower()
    dataset_class = get_dataset_class(dataset_name)
    root = Path(CONFIG["data"]["location"]["root"])
    if dataset_name == "msmt17":
        return dataset_class(root=root, split=split, transform=transform, relabel=relabel)

    split_name = CONFIG["data"]["location"]["splits"][split]
    return dataset_class(root / split_name, transform=transform, relabel=relabel)


def build_dataset_splits(train_transform=None, test_transform=None):
    train_dataset = build_dataset("train", transform=train_transform, relabel=True)
    query_dataset = build_dataset("query", transform=test_transform, relabel=False)
    gallery_dataset = build_dataset("gallery", transform=test_transform, relabel=False)
    return train_dataset, query_dataset, gallery_dataset

In [ ]:
def np_random_choice(items: list[int], size: int) -> list[int]:
    if not items:
        return []
    repeats = math.ceil(size / len(items))
    expanded = items * repeats
    random.shuffle(expanded)
    return expanded[:size]


class RandomIdentitySampler(Sampler[int]):
    def __init__(self, dataset: ImageFolderReIDDataset, batch_size: int, instances_per_identity: int) -> None:
        if batch_size % instances_per_identity != 0:
            raise ValueError("batch_size must be divisible by instances_per_identity")

        self.dataset = dataset
        self.batch_size = batch_size
        self.instances_per_identity = instances_per_identity
        self.identities_per_batch = batch_size // instances_per_identity
        self.index_dic = defaultdict(list)
        self.index_cam_dic = defaultdict(lambda: defaultdict(list))

        for index, label in enumerate(dataset.labels):
            self.index_dic[label].append(index)
            camid = int(dataset.samples[index]["camid"])
            self.index_cam_dic[label][camid].append(index)

        self.pids = list(self.index_dic.keys())
        self.length = self._compute_length()

    def _compute_length(self) -> int:
        total = 0
        for pid in self.pids:
            idxs = self.index_dic[pid]
            num = len(idxs)
            if num < self.instances_per_identity:
                num = self.instances_per_identity
            total += num - num % self.instances_per_identity
        return total

    def __iter__(self):
        batch_indices = []
        pid_to_batches = {}
        for pid in self.pids:
            idxs = self._sample_pid_indices(pid)
            chunks = [
                idxs[i:i + self.instances_per_identity]
                for i in range(0, len(idxs), self.instances_per_identity)
                if len(idxs[i:i + self.instances_per_identity]) == self.instances_per_identity
            ]
            pid_to_batches[pid] = chunks

        available_pids = [pid for pid, chunks in pid_to_batches.items() if chunks]
        while len(available_pids) >= self.identities_per_batch:
            selected_pids = random.sample(available_pids, self.identities_per_batch)
            for pid in selected_pids:
                batch_indices.extend(pid_to_batches[pid].pop(0))
                if not pid_to_batches[pid]:
                    available_pids.remove(pid)
        return iter(batch_indices)

    def __len__(self) -> int:
        return self.length

    def _sample_pid_indices(self, pid: int) -> list[int]:
        idxs = list(self.index_dic[pid])
        if len(idxs) < self.instances_per_identity:
            return np_random_choice(idxs, self.instances_per_identity)

        camera_to_indices = {camid: list(indices) for camid, indices in self.index_cam_dic[pid].items()}
        for indices in camera_to_indices.values():
            random.shuffle(indices)

        sampled_indices = []
        while True:
            available_cams = [camid for camid, indices in camera_to_indices.items() if indices]
            if not available_cams:
                break

            random.shuffle(available_cams)
            group = []
            for camid in available_cams:
                if len(group) >= self.instances_per_identity:
                    break
                group.append(camera_to_indices[camid].pop())

            if len(group) < self.instances_per_identity:
                remaining = [index for indices in camera_to_indices.values() for index in indices]
                while len(group) < self.instances_per_identity and remaining:
                    random.shuffle(remaining)
                    picked = remaining.pop()
                    group.append(picked)
                    for indices in camera_to_indices.values():
                        if picked in indices:
                            indices.remove(picked)
                            break

            if len(group) == self.instances_per_identity:
                sampled_indices.extend(group)
            else:
                break

        return sampled_indices if sampled_indices else np_random_choice(idxs, self.instances_per_identity)


def build_transforms():
    height = CONFIG["data"]["image_height"]
    width = CONFIG["data"]["image_width"]
    augmentation = CONFIG["augmentation"]

    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )

    train_transforms = [
        transforms.Resize((height, width)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Pad(10),
        transforms.RandomCrop((height, width)),
    ]
    if augmentation.get("color_jitter", False):
        train_transforms.append(
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05)
        )
    if augmentation.get("random_affine_degrees", 0.0) > 0.0:
        train_transforms.append(
            transforms.RandomApply([
                transforms.RandomAffine(
                    degrees=augmentation["random_affine_degrees"],
                    translate=(0.03, 0.03),
                    scale=(0.95, 1.05),
                    shear=5,
                )
            ], p=0.4)
        )
    if augmentation.get("random_grayscale_p", 0.0) > 0.0:
        train_transforms.append(transforms.RandomGrayscale(p=augmentation["random_grayscale_p"]))

    train_transforms.extend([transforms.ToTensor(), normalize])
    if augmentation.get("random_erasing", False):
        train_transforms.append(
            transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value="random")
        )
    if augmentation.get("random_occlusion_p", 0.0) > 0.0:
        train_transforms.append(
            transforms.RandomErasing(
                p=augmentation["random_occlusion_p"],
                scale=(0.12, 0.28),
                ratio=(0.8, 1.8),
                value="random",
            )
        )

    train_transform = transforms.Compose(train_transforms)
    test_transform = transforms.Compose([
        transforms.Resize((height, width)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_transform, test_transform

In [ ]:
def build_vit_b16_backbone(pretrained: bool) -> nn.Module:
    weights = ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
    try:
        backbone = vit_b_16(weights=weights)
    except Exception as exc:
        warnings.warn(f"Falling back to random-initialized ViT-B/16: {exc}")
        backbone = vit_b_16(weights=None)
    backbone.heads = nn.Identity()
    return backbone


class ViTReIDModel(nn.Module):
    def __init__(self, num_classes: int, config: dict) -> None:
        super().__init__()
        model_config = config["model"]
        self.backbone = build_vit_b16_backbone(bool(model_config.get("pretrained", True)))
        vit_feature_dim = 768
        embedding_dim = int(model_config["embedding_dim"])
        dropout = float(model_config.get("dropout", 0.1))
        self.use_local_branch = bool(model_config.get("use_local_branch", True))
        self.num_local_stripes = max(2, int(model_config.get("num_local_stripes", 4)))

        self.cls_embedding = nn.Sequential(
            nn.LayerNorm(vit_feature_dim),
            nn.Linear(vit_feature_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.patch_embedding = nn.Sequential(
            nn.LayerNorm(vit_feature_dim),
            nn.Linear(vit_feature_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        if self.use_local_branch:
            self.local_embedding = nn.Sequential(
                nn.LayerNorm(vit_feature_dim * self.num_local_stripes),
                nn.Linear(vit_feature_dim * self.num_local_stripes, embedding_dim),
                nn.BatchNorm1d(embedding_dim),
                nn.GELU(),
                nn.Dropout(float(model_config.get("local_branch_dropout", 0.1))),
            )
            fusion_input_dim = embedding_dim * 3
        else:
            self.local_embedding = None
            fusion_input_dim = embedding_dim * 2

        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def _forward_tokens(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        tokens = self.backbone._process_input(inputs)
        batch_size = tokens.shape[0]
        class_token = self.backbone.class_token.expand(batch_size, -1, -1)
        tokens = torch.cat([class_token, tokens], dim=1)
        encoded = self.backbone.encoder(tokens)
        return encoded[:, 0], encoded[:, 1:]

    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        cls_token, patch_tokens = self._forward_tokens(inputs)
        cls_embedding = self.cls_embedding(cls_token)
        patch_embedding = self.patch_embedding(patch_tokens.mean(dim=1))
        embeddings = [cls_embedding, patch_embedding]
        if self.use_local_branch and self.local_embedding is not None:
            grid_size = int(patch_tokens.size(1) ** 0.5)
            patch_grid = patch_tokens.transpose(1, 2).reshape(
                patch_tokens.size(0),
                patch_tokens.size(2),
                grid_size,
                grid_size,
            )
            stripe_features = torch.flatten(
                nn.functional.adaptive_avg_pool2d(patch_grid, (self.num_local_stripes, 1)),
                1,
            )
            local_embedding = self.local_embedding(stripe_features)
            embeddings.append(local_embedding)
        embedding = self.fusion(torch.cat(embeddings, dim=1))
        logits = self.classifier(embedding)
        return logits, embedding

In [ ]:
class BatchHardTripletLoss(nn.Module):
    def __init__(self, margin: float = 0.3) -> None:
        super().__init__()
        self.margin = margin

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        distances = torch.cdist(embeddings, embeddings, p=2)
        labels = labels.view(-1, 1)
        mask_pos = labels.eq(labels.t())
        mask_neg = ~mask_pos
        eye = torch.eye(mask_pos.size(0), dtype=torch.bool, device=mask_pos.device)
        mask_pos = mask_pos & ~eye
        hardest_pos = distances.masked_fill(~mask_pos, float("-inf")).max(dim=1).values
        hardest_neg = distances.masked_fill(~mask_neg, float("inf")).min(dim=1).values
        valid = mask_pos.any(dim=1) & mask_neg.any(dim=1)
        if not valid.any():
            return embeddings.new_tensor(0.0)
        loss = F.relu(hardest_pos[valid] - hardest_neg[valid] + self.margin)
        return loss.mean()


class CenterLoss(nn.Module):
    def __init__(self, num_classes: int, feat_dim: int) -> None:
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        centers = self.centers.to(embeddings.device)
        labels = labels.to(embeddings.device)
        centers_batch = centers[labels]
        return ((embeddings - centers_batch) ** 2).sum(dim=1).mean()


class ReIDLoss(nn.Module):
    def __init__(self, num_classes: int, embedding_dim: int, config: dict) -> None:
        super().__init__()
        train_config = config["train"]
        self.ce_weight = train_config["ce_weight"]
        self.triplet_weight = train_config["triplet_weight"]
        self.center_loss_weight = train_config.get("center_loss_weight", 0.0)
        self.ce = nn.CrossEntropyLoss(label_smoothing=train_config.get("label_smoothing", 0.0))
        self.triplet = BatchHardTripletLoss(margin=train_config.get("triplet_margin", 0.3))
        self.center = CenterLoss(num_classes=num_classes, feat_dim=embedding_dim) if self.center_loss_weight > 0 else None

    def forward(self, logits: torch.Tensor, embeddings: torch.Tensor, labels: torch.Tensor):
        ce_loss = self.ce(logits, labels)
        triplet_loss = self.triplet(embeddings, labels)
        center_loss = embeddings.new_tensor(0.0)
        total_loss = (self.ce_weight * ce_loss) + (self.triplet_weight * triplet_loss)
        if self.center is not None:
            center_loss = self.center(embeddings, labels)
            total_loss = total_loss + (self.center_loss_weight * center_loss)
        return total_loss, ce_loss, triplet_loss, center_loss

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device, flip_test: bool = False):
    model.eval()
    features = []
    person_ids = []
    camera_ids = []
    paths = []

    for batch in tqdm(loader, desc="Extract", dynamic_ncols=True):
        images = batch["image"].to(device)
        _, embeddings = model(images)
        if flip_test:
            flipped_images = torch.flip(images, dims=[3])
            _, flipped_embeddings = model(flipped_images)
            embeddings = 0.5 * (embeddings + flipped_embeddings)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        features.append(embeddings.cpu())
        person_ids.extend(batch["pid"])
        camera_ids.extend(batch["camid"])
        paths.extend(batch["path"])

    stacked = torch.cat(features, dim=0).numpy()
    return stacked, np.asarray(person_ids), np.asarray(camera_ids), paths


def compute_distance_matrix(query_features: np.ndarray, gallery_features: np.ndarray) -> np.ndarray:
    query_features = query_features / np.linalg.norm(query_features, axis=1, keepdims=True)
    gallery_features = gallery_features / np.linalg.norm(gallery_features, axis=1, keepdims=True)
    return 1 - np.matmul(query_features, gallery_features.T)


def re_rank_distance_matrix(query_features: np.ndarray, gallery_features: np.ndarray, k1: int = 20, k2: int = 6, lambda_value: float = 0.3) -> np.ndarray:
    all_features = np.concatenate([query_features, gallery_features], axis=0).astype(np.float32)
    all_features = all_features / np.linalg.norm(all_features, axis=1, keepdims=True)
    original_dist = 2.0 - 2.0 * np.matmul(all_features, all_features.T)
    original_dist = np.clip(original_dist, 0.0, None)
    original_dist = np.transpose(original_dist / np.maximum(np.max(original_dist, axis=0), 1e-12))
    all_num = original_dist.shape[0]
    query_num = query_features.shape[0]
    v = np.zeros_like(original_dist, dtype=np.float32)
    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)

    for i in range(all_num):
        forward_neighbors = initial_rank[i, : k1 + 1]
        backward_neighbors = initial_rank[forward_neighbors, : k1 + 1]
        reciprocal = forward_neighbors[np.where(backward_neighbors == i)[0]]
        reciprocal_expansion = reciprocal.copy()
        for candidate in reciprocal:
            candidate_forward = initial_rank[candidate, : int(np.around(k1 / 2)) + 1]
            candidate_backward = initial_rank[candidate_forward, : int(np.around(k1 / 2)) + 1]
            candidate_reciprocal = candidate_forward[np.where(candidate_backward == candidate)[0]]
            if len(np.intersect1d(candidate_reciprocal, reciprocal)) > (2.0 / 3.0) * len(candidate_reciprocal):
                reciprocal_expansion = np.append(reciprocal_expansion, candidate_reciprocal)
        reciprocal_expansion = np.unique(reciprocal_expansion)
        weights = np.exp(-original_dist[i, reciprocal_expansion])
        v[i, reciprocal_expansion] = weights / np.sum(weights)

    if k2 > 1:
        v_qe = np.zeros_like(v, dtype=np.float32)
        for i in range(all_num):
            v_qe[i, :] = np.mean(v[initial_rank[i, :k2], :], axis=0)
        v = v_qe

    inv_index = [np.where(v[:, i] != 0)[0] for i in range(all_num)]
    jaccard_dist = np.zeros((query_num, all_num), dtype=np.float32)

    for i in range(query_num):
        temp_min = np.zeros((1, all_num), dtype=np.float32)
        non_zero = np.where(v[i, :] != 0)[0]
        related = [inv_index[idx] for idx in non_zero]
        for j, related_images in enumerate(related):
            temp_min[0, related_images] += np.minimum(v[i, non_zero[j]], v[related_images, non_zero[j]])
        jaccard_dist[i] = 1.0 - temp_min / (2.0 - temp_min)

    final_dist = jaccard_dist * (1 - lambda_value) + original_dist[:query_num, :] * lambda_value
    return final_dist[:, query_num:]


def evaluate_market1501(distance_matrix: np.ndarray, query_pid: np.ndarray, gallery_pid: np.ndarray, query_cam: np.ndarray, gallery_cam: np.ndarray, max_rank: int = 50):
    indices = np.argsort(distance_matrix, axis=1)
    matches = (gallery_pid[indices] == query_pid[:, np.newaxis]).astype(np.int32)
    all_cmc = []
    all_ap = []
    all_inp = []

    for query_idx in range(distance_matrix.shape[0]):
        q_pid = query_pid[query_idx]
        q_cam = query_cam[query_idx]
        order = indices[query_idx]
        remove = (gallery_pid[order] == q_pid) & (gallery_cam[order] == q_cam)
        keep = np.invert(remove)
        raw_cmc = matches[query_idx][keep]
        if not np.any(raw_cmc):
            continue

        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])
        num_rel = raw_cmc.sum()
        precision = raw_cmc.cumsum() / (np.arange(raw_cmc.shape[0]) + 1)
        ap = (precision * raw_cmc).sum() / num_rel
        all_ap.append(ap)
        hardest_match_rank = np.flatnonzero(raw_cmc)[-1] + 1
        inp = num_rel / hardest_match_rank
        all_inp.append(float(inp))

    if not all_cmc:
        raise RuntimeError("No valid query samples were found during evaluation.")

    cmc = np.asarray(all_cmc, dtype=np.float32).mean(axis=0)
    mean_ap = float(np.mean(all_ap))
    mean_inp = float(np.mean(all_inp))
    return cmc, mean_ap, mean_inp, len(all_cmc)

In [ ]:
def build_loaders():
    train_transform, test_transform = build_transforms()
    train_dataset, query_dataset, gallery_dataset = build_dataset_splits(train_transform, test_transform)
    sampler = RandomIdentitySampler(
        train_dataset,
        batch_size=CONFIG["data"]["batch_size"],
        instances_per_identity=CONFIG["data"]["instances_per_identity"],
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["data"]["batch_size"],
        sampler=sampler,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
        drop_last=True,
    )
    query_loader = DataLoader(
        query_dataset,
        batch_size=CONFIG["data"]["eval_batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )
    gallery_loader = DataLoader(
        gallery_dataset,
        batch_size=CONFIG["data"]["eval_batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )
    return train_dataset, train_loader, query_loader, gallery_loader


def build_optimizer(model: nn.Module, criterion: ReIDLoss):
    train_config = CONFIG["train"]
    base_lr = train_config["learning_rate"]
    backbone_lr_factor = train_config.get("backbone_lr_factor", 0.1)
    weight_decay = train_config["weight_decay"]
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)
    parameter_groups = [
        {"params": backbone_params, "lr": base_lr * backbone_lr_factor},
        {"params": head_params, "lr": base_lr},
    ]
    if criterion.center is not None:
        parameter_groups.append({
            "params": criterion.center.parameters(),
            "lr": train_config.get("center_loss_lr", 0.25),
            "weight_decay": 0.0,
        })
    return torch.optim.AdamW(parameter_groups, weight_decay=weight_decay)


def build_scheduler(optimizer):
    train_config = CONFIG["train"]
    scheduler_type = train_config.get("scheduler_type", "cosine").lower()
    if scheduler_type == "plateau":
        return (
            torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="max",
                factor=train_config.get("lr_reduce_factor", 0.5),
                patience=train_config.get("lr_reduce_patience", 5),
                threshold=train_config.get("lr_reduce_threshold", 1e-3),
                min_lr=train_config.get("min_lr", 1e-6),
            ),
            "metric",
        )

    warmup_epochs = train_config.get("warmup_epochs", 0)

    def lr_lambda(epoch: int) -> float:
        total_epochs = max(1, train_config["epochs"])
        if warmup_epochs > 0 and epoch < warmup_epochs:
            return float(epoch + 1) / float(warmup_epochs)
        cosine_epochs = max(1, total_epochs - warmup_epochs)
        progress = (epoch - warmup_epochs) / cosine_epochs
        progress = min(max(progress, 0.0), 1.0)
        min_lr_scale = train_config.get("min_lr_scale", 0.01)
        cosine = 0.5 * (1.0 + torch.cos(torch.tensor(progress * torch.pi)).item())
        return min_lr_scale + (1.0 - min_lr_scale) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda), "epoch"


def save_checkpoint(model, optimizer, scheduler, epoch: int, metrics: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "epoch": epoch,
        "metrics": metrics,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": None if scheduler is None else scheduler.state_dict(),
    }, path)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, use_amp, grad_clip_norm=None):
    model.train()
    running_loss = 0.0
    running_ce = 0.0
    running_triplet = 0.0
    running_center = 0.0
    correct = 0
    total = 0

    progress = tqdm(loader, desc="Train", dynamic_ncols=True)
    for batch in progress:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["pid"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with amp.autocast(device_type=device.type, enabled=use_amp):
            logits, embeddings = model(images)
            loss, ce_loss, triplet_loss, center_loss = criterion(logits, embeddings, labels)

        scaler.scale(loss).backward()
        if grad_clip_norm is not None and grad_clip_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        running_loss += float(loss.item())
        running_ce += float(ce_loss.item())
        running_triplet += float(triplet_loss.item())
        running_center += float(center_loss.item())
        predictions = logits.argmax(dim=1)
        correct += int((predictions == labels).sum().item())
        total += labels.size(0)

        progress.set_postfix(
            loss=f"{running_loss / max(1, progress.n):.4f}",
            acc=f"{100.0 * correct / max(1, total):.2f}%",
        )

    return {
        "train_loss": running_loss / len(loader),
        "train_ce_loss": running_ce / len(loader),
        "train_triplet_loss": running_triplet / len(loader),
        "train_center_loss": running_center / len(loader),
        "train_accuracy": correct / max(1, total),
    }


def run_evaluation(model, query_loader, gallery_loader, device):
    evaluation_config = CONFIG["evaluation"]
    flip_test = evaluation_config.get("flip_test", False)
    query_features, query_pid, query_cam, _ = extract_features(model, query_loader, device, flip_test=flip_test)
    gallery_features, gallery_pid, gallery_cam, _ = extract_features(model, gallery_loader, device, flip_test=flip_test)

    base_distance_matrix = compute_distance_matrix(query_features, gallery_features)
    cmc, mean_ap, mean_inp, valid_queries = evaluate_market1501(
        base_distance_matrix,
        query_pid,
        gallery_pid,
        query_cam,
        gallery_cam,
    )

    results = {
        "rank1": float(cmc[0]),
        "rank5": float(cmc[4]),
        "rank10": float(cmc[9]),
        "rank20": float(cmc[19]),
        "mAP": float(mean_ap),
        "mINP": float(mean_inp),
        "valid_queries": int(valid_queries),
        "rank1_base": float(cmc[0]),
        "rank5_base": float(cmc[4]),
        "rank10_base": float(cmc[9]),
        "rank20_base": float(cmc[19]),
        "mAP_base": float(mean_ap),
        "mINP_base": float(mean_inp),
    }

    if evaluation_config.get("use_rerank", False):
        rerank_distance_matrix = re_rank_distance_matrix(
            query_features,
            gallery_features,
            k1=evaluation_config.get("rerank_k1", 20),
            k2=evaluation_config.get("rerank_k2", 6),
            lambda_value=evaluation_config.get("rerank_lambda", 0.3),
        )
        rerank_cmc, rerank_mean_ap, rerank_mean_inp, _ = evaluate_market1501(
            rerank_distance_matrix,
            query_pid,
            gallery_pid,
            query_cam,
            gallery_cam,
        )
        results.update({
            "rank1_rerank": float(rerank_cmc[0]),
            "rank5_rerank": float(rerank_cmc[4]),
            "rank10_rerank": float(rerank_cmc[9]),
            "rank20_rerank": float(rerank_cmc[19]),
            "mAP_rerank": float(rerank_mean_ap),
            "mINP_rerank": float(rerank_mean_inp),
        })
        results["rank1"] = results["rank1_rerank"]
        results["rank5"] = results["rank5_rerank"]
        results["rank10"] = results["rank10_rerank"]
        results["rank20"] = results["rank20_rerank"]
        results["mAP"] = results["mAP_rerank"]
        results["mINP"] = results["mINP_rerank"]

    return results

## 2. Build loaders and model

In [ ]:
seed_everything(CONFIG["seed"])
device = infer_device(CONFIG["device"])
use_amp = bool(CONFIG["train"]["amp"] and device.type == "cuda")

if device.type != "cuda" and CONFIG["train"]["amp"]:
    warnings.warn("AMP was requested but CUDA is not available. Training will run in FP32.")

train_dataset, train_loader, query_loader, gallery_loader = build_loaders()
model = ViTReIDModel(num_classes=train_dataset.num_classes, config=CONFIG).to(device)
criterion = ReIDLoss(train_dataset.num_classes, CONFIG["model"]["embedding_dim"], CONFIG).to(device)
optimizer = build_optimizer(model, criterion)
scheduler, scheduler_step_mode = build_scheduler(optimizer)
scaler = amp.GradScaler(device.type, enabled=use_amp)

print("Device:", device)
print("Train samples:", len(train_dataset))
print("Num classes:", train_dataset.num_classes)
print("Model built successfully")

## 3. Train and save checkpoints

In [ ]:
train_logger = FileLogger(LOGS_DIR / "train.log")
save_json(CONFIG, LOGS_DIR / "effective_config.json")

best_map = float("-inf")
best_monitored_metric = float("-inf")
bad_epochs = 0
history = []

early_stopping = CONFIG["train"].get("early_stopping", {})
early_stopping_enabled = bool(early_stopping.get("enabled", False))
early_stopping_patience = int(early_stopping.get("patience", 10))
early_stopping_min_delta = float(early_stopping.get("min_delta", 0.0))
early_stopping_monitor = early_stopping.get("monitor", "mAP")

for epoch in range(1, CONFIG["train"]["epochs"] + 1):
    train_metrics = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
        device,
        use_amp,
        grad_clip_norm=CONFIG["train"].get("grad_clip_norm"),
    )
    eval_metrics = run_evaluation(model, query_loader, gallery_loader, device)

    epoch_metrics = {
        "epoch": epoch,
        **train_metrics,
        **eval_metrics,
        "lr": float(optimizer.param_groups[0]["lr"]),
    }
    history.append(epoch_metrics)

    train_logger.log(
        f"Epoch {epoch}/{CONFIG['train']['epochs']} | loss={epoch_metrics['train_loss']:.4f} | "
        f"rank1={epoch_metrics['rank1'] * 100:.2f}% | rank5={epoch_metrics['rank5'] * 100:.2f}% | "
        f"rank10={epoch_metrics['rank10'] * 100:.2f}% | rank20={epoch_metrics['rank20'] * 100:.2f}% | "
        f"mAP={epoch_metrics['mAP'] * 100:.2f}% | mINP={epoch_metrics['mINP'] * 100:.2f}%"
    )

    if CONFIG["evaluation"].get("use_rerank", False):
        train_logger.log(
            f"  Base metrics | rank1={epoch_metrics['rank1_base'] * 100:.2f}% | mAP={epoch_metrics['mAP_base'] * 100:.2f}% | mINP={epoch_metrics['mINP_base'] * 100:.2f}%"
        )
        train_logger.log(
            f"  Rerank metrics | rank1={epoch_metrics['rank1_rerank'] * 100:.2f}% | mAP={epoch_metrics['mAP_rerank'] * 100:.2f}% | mINP={epoch_metrics['mINP_rerank'] * 100:.2f}%"
        )

    last_checkpoint = CHECKPOINTS_DIR / "last_model.pth"
    save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics, last_checkpoint)

    current_monitored_metric = float(epoch_metrics[early_stopping_monitor])
    if scheduler_step_mode == "metric":
        scheduler.step(current_monitored_metric)
    else:
        scheduler.step()

    if epoch_metrics["mAP"] > best_map:
        best_map = epoch_metrics["mAP"]
        best_checkpoint = CHECKPOINTS_DIR / "best_model.pth"
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics, best_checkpoint)

    if current_monitored_metric > (best_monitored_metric + early_stopping_min_delta):
        best_monitored_metric = current_monitored_metric
        bad_epochs = 0
    else:
        bad_epochs += 1

    if early_stopping_enabled and bad_epochs >= early_stopping_patience:
        train_logger.log(
            f"Early stopping triggered at epoch {epoch} after {bad_epochs} epochs without improvement in {early_stopping_monitor}."
        )
        break

summary = {
    "run_slug": RUN_NAME,
    "run_root": str(ARTIFACT_ROOT),
    "dataset": CONFIG["data"]["dataset"]["name"],
    "best_epoch": max(history, key=lambda item: item["mAP"])["epoch"],
    "best_rank1": max(history, key=lambda item: item["mAP"])["rank1"],
    "best_rank5": max(history, key=lambda item: item["mAP"])["rank5"],
    "best_rank10": max(history, key=lambda item: item["mAP"])["rank10"],
    "best_rank20": max(history, key=lambda item: item["mAP"])["rank20"],
    "best_mAP": max(history, key=lambda item: item["mAP"])["mAP"],
    "best_mINP": max(history, key=lambda item: item["mAP"])["mINP"],
    "history": history,
}

save_json(summary, METRICS_DIR / "metrics_v1.json")
print("Training completed")

## 4. Evaluate best checkpoint and save evaluation log

In [ ]:
evaluate_logger = FileLogger(LOGS_DIR / "evaluate.log")
best_checkpoint = CHECKPOINTS_DIR / "best_model.pth"
if not best_checkpoint.exists():
    raise FileNotFoundError(f"Missing checkpoint: {best_checkpoint}")

checkpoint = torch.load(best_checkpoint, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
results = run_evaluation(model, query_loader, gallery_loader, device)

evaluation_payload = {
    "run_slug": RUN_NAME,
    "run_root": str(ARTIFACT_ROOT),
    "dataset": CONFIG["data"]["dataset"]["name"],
    "checkpoint": str(best_checkpoint),
    "loaded_epoch": checkpoint.get("epoch"),
    **results,
}

save_json(evaluation_payload, METRICS_DIR / "evaluation_latest.json")
evaluate_logger.log(json.dumps(evaluation_payload, indent=2))
print("Evaluation completed")

## 5. Output summary

Sau khi chay xong, hay download thu muc artifact trong `/kaggle/working/artifacts/...`.

Nhung file quan trong:
- `checkpoints/best_model.pth`
- `checkpoints/last_model.pth`
- `metrics/metrics_v1.json`
- `metrics/evaluation_latest.json`
- `logs/train.log`
- `logs/evaluate.log`
- `logs/effective_config.json`

In [ ]:
print("Artifact root:", ARTIFACT_ROOT)
for path in sorted(ARTIFACT_ROOT.rglob("*")):
    print(path)